# WHOMOLLM Project
## Geo context prompt (Version 3) 22.Feb.2026 based on the Qwen 3.1 model feature: household income level
1. Raw data
2. stay segmentation
3. reverse geocoding
4. top 5 POIs aggregation
5. semantic stay table
6. LLM prompt (daily narrative/ activity inference/ other surplus info)

## 1. Test GPU, Qwen model 

In [14]:
import os
import json
import re
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
# ===============================
# GPU CHECK
# ===============================

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM (GB):", round(p.total_memory / 1024**3, 2))

# ===============================
# LOAD QWEN MODEL (4bit for L4)
# ===============================

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

print("Loading Qwen...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()
print("Qwen ready.")

CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.06
Loading Qwen...


Loading weights: 100%|██████████| 339/339 [00:02<00:00, 155.22it/s, Materializing param=model.norm.weight]                              


Qwen ready.


## 2. Generate prompts (3nd version with just geo context information and behaviour info, predict household size and household income level ... for all users used different model, Qwen)
Version control:
This is the basic prompt based on geocontext and behaviour info, version 3.

In [15]:
#   ---------------------------
# Version 3 (22 Feb 2026) prompt with geo context to predict features
#   ---------------------------

import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb

# ---------------------------
# Config
# ---------------------------
RANDOM_SEED = 42
N_USERS     = 5
MAX_EVENTS  = 300
HOUR_BIN    = 1
PREC        = int(os.getenv("COORD_PREC", "4"))


DATA_SP     = Path("/data/baliu/python_code/data/sp_all copy.csv")
PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v3_features_22Feb2026.json")

# -------------------------------------------   
# Load (strings) + base cleaning
# -------------------------------------------
sp = pd.read_csv(DATA_SP, dtype=str, engine="python", on_bad_lines="skip")
print("sp shape:", sp.shape)

# Datetimes
sp["started_at"]  = pd.to_datetime(sp["started_at"], errors="coerce", utc=True)
sp["finished_at"] = pd.to_datetime(sp.get("finished_at"), errors="coerce", utc=True)

# Drop unusable
sp = sp.dropna(subset=["user_id", "started_at", "location_id"]).copy()
sp["user_id"]     = sp["user_id"].astype(str)
sp["location_id"] = sp["location_id"].astype(str)

#sp["duration_min"] = ((sp["finished_at"] - sp["started_at"]).dt.total_seconds() / 60.0).astype(float)
print(sp["duration"].head())
print(sp["duration"].dtype)

# Force numeric
sp["act_duration"] = pd.to_numeric(
    sp["act_duration"],
    errors="coerce"
)

# change it to hours
sp["act_duration_h"] = (sp["act_duration"] / 60.0).round(1)
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)

# length_km: convert to numeric, coerce errors to NaN, then divide by 1000
sp["length_km"] = pd.to_numeric(sp["length"], errors="coerce") / 1000.0
sp["length_km"] = sp["length_km"].round(1)

# Time features
sp["date"]     = sp["started_at"].dt.date
sp["dow"]      = sp["started_at"].dt.dayofweek.astype(int)        # 0=Mon
sp["hour_bin"] = (sp["started_at"].dt.hour // HOUR_BIN * HOUR_BIN).astype(int)

# ---------------------------
# Format from wkt point -> lon/lat (rounded)
# --------------------------------
WKT_POINT_RE = re.compile(
    r"POINT\s*\(\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*\)",
    re.I,
)

def extract_lonlat_from_wkt(s: str):
    m = WKT_POINT_RE.search(str(s))
    if not m: return (np.nan, np.nan)
    return float(m.group(1)), float(m.group(2))

sp["geometry"] = sp.get("geometry", pd.Series(index=sp.index)).astype(str)
lonlat = sp["geometry"].apply(extract_lonlat_from_wkt)
sp[["lon","lat"]] = pd.DataFrame(lonlat.tolist(), index=sp.index)
sp["lon"] = pd.to_numeric(sp["lon"], errors="coerce").round(PREC)
sp["lat"] = pd.to_numeric(sp["lat"], errors="coerce").round(PREC)

# Lightweight tags
sp["geometry_type"] = np.where(sp["lon"].notna() & sp["lat"].notna(), "point", "-")
if "mode" not in sp.columns: sp["mode"] = "-"

sp shape: (1152987, 10)
0    1093.0
1      39.0
2      75.0
3     164.0
4      36.0
Name: duration, dtype: str
str
0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64


In [16]:
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)
print(sp["length_km"].head())
print(sp["length_km"].dtype)

0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64
0     0.0
1    11.6
2     2.1
3     4.8
4    12.6
Name: length_km, dtype: float64
float64


In [17]:
print ("sp after cleaning:", sp.shape)
print (sp.head(5))

sp after cleaning: (862871, 18)
  id user_id                       started_at  \
0  0   AAGAF 2019-10-09 11:30:34.141000+00:00   
1  1   AAGAF 2019-10-10 06:14:49.141999+00:00   
2  2   AAGAF 2019-10-10 07:03:24.426000+00:00   
3  3   AAGAF 2019-10-10 11:10:24.605999+00:00   
4  4   AAGAF 2019-10-10 14:30:45.187999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-10 05:43:17.674999+00:00           0   Car                 0.0   
1 2019-10-10 06:53:54.841000+00:00           1   Car  11615.408548376423   
2 2019-10-10 08:18:20.864000+00:00           2   Car  2104.8558581266784   
3 2019-10-10 13:54:34.799339+00:00           2  Walk   4847.706521391238   
4 2019-10-10 15:07:07.127239+00:00           3   Car  12621.909934521313   

                                       geometry duration  act_duration  \
0  POINT (7.565219245342489 47.545616372908086)   1093.0        1093.0   
1  POINT (7.5637597959179645 47.54794767256367)     39.0          71

In [18]:
# ---------------------------
# Sample 1 week per user
# ---------------------------
def sample_one_week_per_user(sp, seed=42):
    rng = np.random.default_rng(seed)
    sp = sp.copy()
    sp["date"] = pd.to_datetime(sp["date"])

    out = []
    for uid, df_u in sp.groupby("user_id"):
        days = (
            df_u[["date"]]
            .drop_duplicates()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if len(days) < 7:
            continue
        days["delta"] = days["date"].diff().dt.days.fillna(1).astype(int)
        days["block"] = (days["delta"] >1).cumsum()

        valid_blocks = (
            days.groupby("block")
            .filter(lambda x: len(x) >=7)
        )

        if valid_blocks.empty:
            continue
        candidate_starts = []
        for _, g in valid_blocks.groupby("block"):
            block_dates = g["date"].sort_values().reset_index(drop=True)
            for i in range(len(block_dates) - 6):
                candidate_starts.append(g.iloc[i]["date"])

        start_date = rng.choice(candidate_starts)
        week_dates = pd.date_range(start=start_date, periods=7, freq= 'D')

        out.append(
            df_u[df_u["date"].isin(week_dates)]

        )
       
    return pd.concat(out, ignore_index=True)

sp_sampled2 = sample_one_week_per_user(sp, seed=RANDOM_SEED)
print("sp_sampled2 shape:", sp_sampled2.shape)
print(sp_sampled2.head(2))

# ---------------------------
# save sampled data
# ---------------------------
SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_1week_22Feb2026_qwen.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)
print("Saved sp_sampled2_1week_22Feb2026 to", SP_SAMPLED2_OUT)  

sp_sampled2 shape: (60738, 18)
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   

                       finished_at location_id  mode             length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus  3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram  8232.492568529495   

                                       geometry duration  act_duration  \
0  POINT (7.564359990583688 47.546459921667946)   1097.0        6456.0   
1   POINT (7.603269084785703 47.58100341208616)    116.0         275.0   

   act_duration_h  length_km       date  dow  hour_bin     lon      lat  \
0           107.6        3.8 2019-10-22    1         9  7.5644  47.5465   
1             4.6        8.2 2019-10-23    2         6  7.6033  47.5810   

  geometry_type  
0         point  
1         point  
Saved sp_sampled2_1week_22Feb2026 to /data/baliu/python_code/data/sp_sampled2_1week_22Feb

#### Load location name and POIs top 5

In [19]:
# 1. Load toponym data from shepefiles
import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb
import geopandas as gpd
import fiona
# read parquet file
import pyarrow.parquet as pq
 
CACHE_DIR   = Path("/data/baliu/python_code/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NOM_CACHE_PATH = CACHE_DIR / "nominatim_cache.parquet"
nom = pd.read_parquet(NOM_CACHE_PATH)
print("Nominatim cache loaded:", nom.shape)
print("Nominatim columns:", nom.columns.tolist())
nom.head(30)
groupby_category = nom.groupby('city').size()
print(groupby_category) 

Nominatim cache loaded: (370983, 6)
Nominatim columns: ['lon', 'lat', 'road', 'neighbourhood', 'city', 'postcode']
city
A Coruña                 2
A Illa de Arousa         2
A Pobra do Caramiñal     7
A Seara                  1
Aachen                   2
                        ..
دهستان سروستان           1
دهستان فجر               3
دهستان کرکس              1
บ้านอ่าวน้ำเมา           1
เทศบาลนครเกาะสมุย       55
Length: 7400, dtype: int64


In [20]:
from notebook_utils.geocontext import attach_nominatim, clean_nominatim_fields

sp_sampled2 = attach_nominatim(sp_sampled2, nom)
sp_sampled2 = clean_nominatim_fields(sp_sampled2)
print("sp_sampled2 after attaching nominatim:", sp_sampled2.shape)
print("sp_sampled2 columns:", sp_sampled2.columns.tolist())
print(sp_sampled2.head(10))


sp_sampled2 after attaching nominatim: (60738, 22)
sp_sampled2 columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode']
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   
5  28   AAGAF 2019-10-24 06:05:44.141999+00:00   
6  29   AAGAF 2019-10-24 07:22:44.510999+00:00   
7  30   AAGAF 2019-10-24 09:55:47.243000+00:00   
8  31   AAGAF 2019-10-24 13:05:05.668999+00:00   
9  32   AAGAF 2019-10-24 15:03:21.792999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10 

#### Build token with geocontext info

In [21]:
from pathlib import Path
from notebook_utils.geocontext import load_poi_frame

POI_PATH = Path("/data/baliu/python_code/data/version2/data/final_pois_nob.gpkg")
poi = load_poi_frame(POI_PATH)
pois = poi
poi.head()


,id,code,name,category,geometry
0,0,3100,Temple de Saint-Jean,Civic,POINT (2498816.948 1117839.539)
1,1,3100,Kapelle Oberes Heiliges Kreuz,Civic,POINT (2691857.357 1192677.631)
2,2,3102,Kirche St. Johannes,Civic,POINT (2599659.594 1202970.041)
3,3,3102,Paroisse catholique,Civic,POINT (2501879.911 1126418.086)
4,4,3300,Mosquée du Petit-Saconnex,Civic,POINT (2498411.634 1119984.893)


In [22]:
poi = poi.copy()
print(poi["category"].value_counts())
print(f"poi crs: {poi.crs}")


category
Others            158579
Unknown           114894
Entertainment     106734
Shopping           54634
Residential        45556
Transportation     40997
Services            9525
Schools             2904
Civic                780
Name: count, dtype: int64
poi crs: EPSG:2056


In [23]:
from notebook_utils.geocontext import load_poi_frame

def load_pois_once():
    global poi
    if poi is None:
        print("Loading POIs into memory...")
        poi = load_poi_frame(POI_PATH)
    return poi

print(poi.head(2))


   id  code                           name category  \
0   0  3100           Temple de Saint-Jean    Civic   
1   1  3100  Kapelle Oberes Heiliges Kreuz    Civic   

                          geometry  
0  POINT (2498816.948 1117839.539)  
1  POINT (2691857.357 1192677.631)  


In [24]:
from notebook_utils.geocontext import build_location_gdf

loc = build_location_gdf(sp_sampled2)


### update the table with places name and pois

update

In [25]:
# update: 
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree

def bearing_to_direction(dx, dy):
    angle = np.degrees(np.arctan2(dx, dy))
    angle = (angle + 360) % 360

    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

def get_poi_context_for_prompt(sp_sampled2, poi, top_k=5):

    # ---- prepare GeoDataFrames ----
    loc = gpd.GeoDataFrame(
        sp_sampled2.copy(),
        geometry=gpd.points_from_xy(
            sp_sampled2["lon"].astype(float),
            sp_sampled2["lat"].astype(float)
        ),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi.copy().set_crs(epsg=2056, allow_override=True)

    # ---- KDTree on POIs ----
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]
    dists, idxs = tree.query(loc_xy, k=top_k * 5) # get more to filter later

    # ---- build kNN table ----
    rows = []
    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name": poi.iloc[j]["name"],
                "category": poi.iloc[j]["category"],
                "addr_poi_dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })

    joined = pd.DataFrame(rows)

    # ---- clean categories ----
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    # ---- distance + direction ----
    joined["addr_poi_dist_km"] = (joined["addr_poi_dist_m"] / 1000).round(3)
    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]
    
    joined = (
        joined
        .sort_values("addr_poi_dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )

    return joined[
        ["location_id", "name", "category", "addr_poi_dist_km", "direction"]
    ]

def clean_text_part(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower() in ["none", "nan", "-", ""]:
        return None
    return s

def build_poi_prompt_text(df):
    out = []
    for _, r in df.iterrows():
        dist = r["addr_poi_dist_km"]
        direction = r["direction"].lower()
        
       
        category = clean_text_part(r.get("category")) or "point of interest"
        name = clean_text_part(r.get("name"))

        if name:
            poi_desc = f"{category} {name}"
        else:
            poi_desc = f"{category}"
        out.append(
            f"{dist} km to the {direction}: {poi_desc}"
        )
    return out

poi_prompt_df = (
    get_poi_context_for_prompt(sp_sampled2, poi, top_k=5)
    .groupby("location_id", group_keys=False)
    .apply(build_poi_prompt_text, include_groups=False)
    .reset_index(name="nearby_places")
)

sp_sampled2 = sp_sampled2.merge(
    poi_prompt_df,
    on="location_id",
    how="left"
)
print(
    get_poi_context_for_prompt(sp_sampled2, poi)
    ["addr_poi_dist_km"]
    .describe()
)

from pathlib import Path

SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_v3_1week_22Feb2026qwen_with_geocontext.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)

print("Saved sp_sampled2 to", SP_SAMPLED2_OUT)
print(sp_sampled2.head(5))

count    116301.000000
mean        288.675402
std        1458.250814
min           0.000000
25%           0.042000
50%           0.101000
75%           0.262000
max       14837.215000
Name: addr_poi_dist_km, dtype: float64
Saved sp_sampled2 to /data/baliu/python_code/data/sp_sampled2_v3_1week_22Feb2026qwen_with_geocontext.csv
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816718+00:00          10   Car  3283.14361600

In [26]:
# update: 
import numpy as np

def bearing_to_direction(dx, dy):
    angle = np.degrees(np.arctan2(dx, dy))
    angle = (angle + 360) % 360

    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree

def get_poi_context_for_prompt(sp_sampled2, poi, top_k=5):

    # ---- prepare GeoDataFrames ----
    loc = gpd.GeoDataFrame(
        sp_sampled2.copy(),
        geometry=gpd.points_from_xy(
            sp_sampled2["lon"].astype(float),
            sp_sampled2["lat"].astype(float)
        ),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi.copy().set_crs(epsg=2056, allow_override=True)

    # ---- KDTree on pois ----
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]
    dists, idxs = tree.query(loc_xy, k=top_k * 3) # get more to filter later

    # ---- build kNN table ----
    rows = []
    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name": poi.iloc[j]["name"],
                "category": poi.iloc[j]["category"],
                "addr_poi_dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })

    joined = pd.DataFrame(rows)

    # ---- clean categories ----
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    # ---- distance + direction ----
    joined["addr_poi_dist_km"] = (joined["addr_poi_dist_m"] / 1000).round(3)
    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]
    
    joined = (
        joined
        .sort_values("addr_poi_dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )

    return joined[
        ["location_id", "name", "category", "addr_poi_dist_km", "direction"]
    ]

def build_poi_prompt_text(df):
    out = []
    for _, r in df.iterrows():
        name = r["name"] if pd.notna(r["name"]) else "unknown place"
        category = r["category"] if pd.notna(r["category"]) else "point of interest"
        out.append(
            f"{r['addr_poi_dist_km']} km to the {r['direction'].lower()}: {category} {name}"
        )
    return "; ".join(out)

poi_prompt_df = (
    get_poi_context_for_prompt(sp_sampled2, poi, top_k=5)
    .groupby("location_id", group_keys=False)
    .apply(build_poi_prompt_text, include_groups=False)
    .reset_index(name="nearby_places")
)

sp_sampled2 = sp_sampled2.merge(
    poi_prompt_df,
    on="location_id",
    how="left"
)
print(
    get_poi_context_for_prompt(sp_sampled2, poi)
    ["addr_poi_dist_km"]
    .describe()
)

print("nearby_places in columns",
      "nearby_places" in sp_sampled2.columns)

print(
    sp_sampled2["nearby_places"]
    .notna()
    .value_counts(dropna=False)
)

print(sp_sampled2[["location_id", "nearby_places"]].head(5))


from pathlib import Path

SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_v3_with_geocontext_qwen.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)

print("Saved sp_sampled2 to", SP_SAMPLED2_OUT)
print(sp_sampled2.head(5))

KeyboardInterrupt: 

same as above, upgraded 

In [27]:
from notebook_utils.geocontext import print_poi_feature_summary

print_poi_feature_summary(sp_sampled2)



Final sp_sampled2 with POI features:
Columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode', 'nearby_places']

Sample rows:
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816718+00:00          10   Car  3283.1436

In [28]:
import ast
import pandas as pd

def tokens_compact_1week(
    df_u: pd.DataFrame,
    max_events=50,
    prec=4
) -> list[str]:
    """
    Build natural-language mobility tokens from up to one week of staypoints,
    inserting a date header at the beginning of each day.
    """

    # --------------------------------------------------
    # 1. detect duration column
    # --------------------------------------------------
    duration_col = None
    for c in df_u.columns:
        if c.lower() in ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]:
            duration_col = c
            break

    # --------------------------------------------------
    # 2. select usable columns
    # --------------------------------------------------
    use_cols = [
        "started_at", "dow", "hour_bin", "location_id",
        "lon", "lat", "mode",
        "city", "neighbourhood", "road", "nearby_places",
        "postcode"
    ]
    if duration_col:
        use_cols.append(duration_col)

    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()

    # --------------------------------------------------
    # 3. datetime handling + ordering + safety cap
    # --------------------------------------------------
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at")

    if len(df) > max_events:
        df = df.head(max_events)

    # --------------------------------------------------
    # 4. build tokens with date headers
    # --------------------------------------------------
    toks = []
    current_date = None

    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()

        # ---- insert date header when date changes ----
        if date_str != current_date:
            toks.append(f"Date: {date_str}")
            current_date = date_str

        # ---- time info ----
        hhmm = t.strftime("%H:%M")
        dow = int(getattr(r, "dow", -1))
        hb  = int(getattr(r, "hour_bin", -1))

        # ---- location ----
        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)

        def clean_addr_part(s):
            if pd.isna(s):
                return None
            s = str(s).strip()
            if s.lower() in ["nan", "none", "-", ""]:
                return None
            return s
        addr_parts = [
            clean_addr_part(getattr(r, "road", None)),
            clean_addr_part(getattr(r, "neighbourhood", None)),
            clean_addr_part(getattr(r, "city", None)),
        ]
        addr_parts = [p for p in addr_parts if p is not None]
        addr = ", ".join(addr_parts) if addr_parts else "unknown address"
        # add postcode
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode is not None:
            addr = f"{addr}, postcode {postcode}"
        #addr = addr.append("postcode " + str(getattr(r, "postcode", "unknown"))) if hasattr(r, "postcode") else addr


        # # ---- nearby POIs ----
        # nearby_raw = getattr(r, "nearby_places", None)
        # if isinstance(nearby_raw, str):
        #     try:
        #         nearby = ast.literal_eval(nearby_raw)
        #     except (ValueError, SyntaxError):
        #         nearby = []
        # elif isinstance(nearby_raw, list):
        #     nearby = nearby_raw
        # else:
        #     nearby = []

        # nearby_str = "; ".join(nearby[:5]) if nearby else "no nearby places listed"

        # ---- nearby POIs ----
        nearby_raw = getattr(r, "nearby_places", None)

        if isinstance(nearby_raw, list):
        # legacy case: list of strings
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str):
         # current correct case: already a string
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"


        # ---- mode & duration ----
        mode = str(getattr(r, "mode", "unknown")).lower()
        dur_val = getattr(r, duration_col, 0) if duration_col else 0
        dur = int(round(float(dur_val))) if pd.notna(dur_val) else 0

        # ---- final sentence ----
        toks.append(
            f"At {hhmm}, on weekday {dow} (hour block {hb}), "
            f"the user was at coordinates ({lat}, {lon}). "
            f"The address is {addr}. "
            f"The transport mode was {mode}, with an activity duration of {dur} minutes. "
            f"Nearby places include: {nearby_str}."
        )

    return toks

In [29]:
toks = tokens_compact_1week(
    df_u=sp_sampled2,
    max_events=40,
    prec=4
)

print(toks[:8])

['Date: 2019-09-02', 'At 20:21, on weekday 0 (hour block 20), the user was at coordinates (46.936, 7.4314). The address is Chutzenstrasse, Bern, postcode 3007. The transport mode was walk, with an activity duration of 503 minutes. Nearby places include: 0.054 km to the north-west: Residential Bahnhof Weissenbühl; 0.06 km to the north-east: Shopping Nido; 0.062 km to the west: Shopping; 0.062 km to the west: Shopping; 0.064 km to the west: Transportation Weissenbühl Bahnhof.', 'Date: 2019-09-03', 'At 05:24, on weekday 1 (hour block 5), the user was at coordinates (47.2148, 7.5201). The address is Vogelherdstrasse, Roamer, postcode 4513. The transport mode was car, with an activity duration of 615 minutes. Nearby places include: 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.088 km to the

### Update prompts Version 3 22 Feb 2026

In [ ]:
# ---------------------------
# Build prompts.  update on 22 Feb 2026 
# ------------------------

from textwrap import dedent

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
sp_prompt = sp_sampled2.copy()
sp_prompt["date_str"] = (
    pd.to_datetime(sp_prompt["started_at"], errors="coerce")
      .dt.tz_localize(None)
      .dt.date
      .astype(str)
)
print(sp_prompt[["started_at", "date", "date_str"]].head(5))

# build valid user-day table from EXISTING rows only
user_day = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
)
user_day = (
    user_day
    .groupby("user_id", group_keys=False)
    #.sample(n=2, random_state=RANDOM_SEED)
    #.rename(columns={"date_str": "sample_date"})
)

#print("user_day size:", len(user_day))   # MUST be > 0

rows_prompts = []

user_dates = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
    .groupby("user_id")["date_str"]
    .apply(list)
)

print(f"user_dates sample: {len(user_dates)}")

for user_id, dates in user_dates.items():
    
    df_u = sp_prompt[

        (sp_prompt["user_id"] == user_id) &
        (sp_prompt["date_str"].isin(dates))
    ]
    print(f"Processing user {user_id} for dates {dates}, df_u shape: {df_u.shape}")


    toks = tokens_compact_1week(df_u, max_events=MAX_EVENTS, prec=PREC)
    if not toks:
        continue
    prompt = (
        f"User: {user_id}\n\n"
        f"Mobility record for one week in Switzerland:\n"
        "Task:\n"
        "Based only on the mobility behaviour and nearby spatial context described below,"
        "infer the user's most likely household income level based on swiss census data and situation\n\n"
        "Please choose exactly one of the following categories (monthly household income in CHF):\n"
        "<4000, 4001-8000, 8001-12000, 12001-16000, 16001+.\n\n"
        "Guidelines:\n"
        "-Use only the information explicitly provided in the mobility record.\n"
        "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
        "- Do not assume any unobserved personal attributes.\n"
        "- keep a neutral and factual tone.\n\n"
        "Output format:\n"
        "1. Reasoning (no more than 100 words).\n"
        "2. One final line in strict JSON format, for example:\n"
        " household_income_level: 4001-8000.\n\n"
        "Mobility evidance:\n"
        + "\n".join(toks)
    )
    
    # store results as a dictionary
    rows_prompts.append({
        "user_id": user_id,
        "date": dates,
        "prompt": prompt})

PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_update_v2.json")

PROMPTS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(PROMPTS_OUT, "w", encoding="utf-8") as f:
    for r in rows_prompts:
        f.write(r["prompt"])
        f.write("\n\n" + "="*80 + "\n\n")

print(f"Saved {len(rows_prompts)} prompts -> {PROMPTS_OUT}")

# print sample
for r in rows_prompts[:1]:
    print("\n--- Sample Prompt ---")
    print(r["prompt"][:1000] + "\n...")   

                        started_at       date    date_str
0 2019-10-22 09:39:28.773999+00:00 2019-10-22  2019-10-22
1 2019-10-23 06:35:32.236999+00:00 2019-10-23  2019-10-23
2 2019-10-23 09:02:58.773999+00:00 2019-10-23  2019-10-23
3 2019-10-23 09:53:18.151999+00:00 2019-10-23  2019-10-23
4 2019-10-23 16:44:43.608999+00:00 2019-10-23  2019-10-23
user_dates sample: 2102
Processing user AAGAF for dates ['2019-10-22', '2019-10-23', '2019-10-24', '2019-10-25', '2019-10-26', '2019-10-27', '2019-10-28'], df_u shape: (23, 24)
Processing user AAINS for dates ['2019-11-13', '2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17', '2019-11-18', '2019-11-19'], df_u shape: (34, 24)
Processing user AAQME for dates ['2019-12-10', '2019-12-11', '2019-12-12', '2019-12-13', '2019-12-14', '2019-12-15', '2019-12-16'], df_u shape: (23, 24)
Processing user AARWP for dates ['2019-09-18', '2019-09-19', '2019-09-20', '2019-09-21', '2019-09-22', '2019-09-23', '2019-09-24'], df_u shape: (35, 24)
Processing user 

Clean the prompts 

In [ ]:
import json
import re
from pathlib import Path

IN_PATH  = Path("/data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_update_v2.json")
OUT_PATHh = Path("/data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2.txt")
# prompts_v3_1week_income_22Feb2026_qwen_update_v2

def clean_prompt_text(text: str) -> str:
    """
    Remove markdown headers, separators, examples, and keep
    only instruction + mobility evidence.
    """
    # 1. remove separator lines ---
    text = re.sub(r"^\s*---\s*$", "", text, flags=re.MULTILINE)

    # 2. remove markdown headers like ## Task Overview
    text = re.sub(r"^\s*## .*?$", "", text, flags=re.MULTILINE)

    # 3. remove Example block
    text = re.sub(
        r'## Example:.*?---',
        '',
        text,
        flags=re.DOTALL
    )

    # 4. collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

cleaned_rows = []

with open(IN_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

blocks = [b.strip() for b in re.split(r"\n{2,}={80,}\n{2,}", raw) if b.strip()]

# apply cleaning
cleaned_blocks = [clean_prompt_text(b) for b in blocks if clean_prompt_text(b)]

# write cleaned jsonl
with open(OUT_PATHh, "w", encoding="utf-8") as f:
    for b in cleaned_blocks:
        f.write(b)
        f.write("\n\n"+ "="*80 + "\n\n")

print(f"Cleaned prompts saved: {len(cleaned_blocks)} -> {OUT_PATHh}")

Cleaned prompts saved: 2102 -> /data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2.txt


## 3. Predict Sociodemographics using "Qwen" model based on the prompts we just generated 

#### Test GPU, decide Model version/ Model name without finetuing verson

In [5]:
%pip install -U "transformers" "accelerate" "safetensors" "tokenizers"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 54.5 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Note: you may need to restart the kernel to use updated packages.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Proper 4-bit config
bnb_config = BitsAndBytesConfig( # the parameter are actually in bitandbytescofig, not in the model loading.
    # all the parameters are for 4bit .
    # we can try later when doing the fine tuning.
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained( # the parameters are not belong to here, that's why we need to remove them, 
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("✅ Qwen ready.")

ModuleNotFoundError: Could not import module 'Qwen2ForCausalLM'. Are this object's requirements defined correctly?

In [3]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import pandas as pd
import torch




# --------------------------------------
# Config
# --------------------------------------

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v6withrationale.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v6withrationale.csv")
# data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2

SEP = "=" * 80

MAX_PROMPT_CHARS = 30000
# MAX_PROMPT_CHARS = 30000  # Qwen 7B can handle up to 32k tokens, which is roughly 30k chars, avoid OOM

TIMEOUT_SEC = 600
COOLDOWN_EVERY = 20
COOLDOWN_SEC = 10

# --------------------------------------
# Timeout handling
# --------------------------------------

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# Load done users
# --------------------------------------
def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------
with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

# --------------------------------------
# Main loop
# --------------------------------------
try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        if len(prompt) > MAX_PROMPT_CHARS:
            print("⚠️ Prompt too long, skipping")
            continue

        try:
            signal.alarm(TIMEOUT_SEC)

            messages =[
                {"role": "system",
                 "content": (
                     "You are a mobility behavior and socioeconomic status inference analyst."
                     "return only valid JSON with keys: age_group, gender, household_size, household_income_level."
                     "Based only on the mobility behaviour and nearby spatial context described below,"
                     # "infer the user's most likely household income level based on swiss census data and situation\n\n"
                     "Please choose exactly one of the following categories (household size, monthly household income in CHF),age_group, gender:\n"
                     "household size: 1, 2,3, 4+, hosuehold income: <4000, 4001-8000, 8001-12000, 12001-16000, 16001+. age: 0-17, 18-24, 25-34, 35-44, 45-54, 55-64, 65+. gender: male, female, other, perefer not to say\n\n"
                     " Guidelines:\n"
                     "- Use only the information explicitly provided in the mobility record.\n"
                     "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
                     "- Do not assume any unobserved personal attributes.\n"
                     "- keep a neutral and factual tone.\n\n"
                     "Output format:\n"
                     "1. Reasoning (no more than 150 words).\n" # need to adjust the parameters for rationale length
                     "2. One final line in strict JSON format, for example:\n"
                     " {\"household_income_level\": \"4001-8000\", \"age_group\": \"45-54\", \"gender\": \"other\", \"income_rationale\": [\"visited high-income neighborhoods\", \"used premium transport modes\", \"frequent visits to business districts\"], \"income_confidence\": \"high\"}\n\n"
                     "- income_rationale must be a list of 3-6 short bullet-like strings, each describing a concrete mobility/semantic evidence. \n "
                     "- Do not provide step by step reasoning, or hidden chain of thought"
                     " Just provide evidence-style justification.\n"
                     "- income_confidence must be one of :low, medium, high.\n"
                     "no extra keys, no markdown, no commentary outside josn"

                 )
                 },
            ]

            tokenizer = AutoTokenizer.from_pretrained(
                MODEL_NAME,
                trust_remote_code=True)

            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=512, # give space for rationale
                temperature=0.5,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.05 # to encourage more diverse outputs
            )

            signal.alarm(0)
            
            # decode only the generated part, avoid mixing prompt + answer
            gen_ids = outputs[0][inputs["input_ids"].shape[1]:]  

            decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            json_match = re.search(r"\{.*\}", decoded, re.DOTALL)
            if json_match:
                #json_str = json_match.group(0)
                #try:
                    #result = json.loads(json_str)


                # need to fix 
                result = json.loads(json_match.group(0))
            else:
                result = {"raw_output": decoded, "error": "json_parse_failed"}

            result["user_id"] = user_id

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done_users.add(user_id)
            print("✅ Done")

        except TimeoutError:
            print("⏰ Timeout")
            continue

        except Exception as e:
            print("❌ Error:", e)
            time.sleep(2)
            continue

        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted safely.")
    sys.exit(0)

# --------------------------------------
# Save csv snapshot
# --------------------------------------

if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved → {PRED_CSV}")
else:
    print("⚠️ No predictions generated")

📦 Loaded 2102 prompts
🔁 Found 0 completed users

🔮 [1/2102] Predicting AAGAF
❌ Error: name 'AutoTokenizer' is not defined

🔮 [2/2102] Predicting AAINS
❌ Error: name 'AutoTokenizer' is not defined

🔮 [3/2102] Predicting AAQME
❌ Error: name 'AutoTokenizer' is not defined

🛑 Interrupted safely.


SystemExit: 0

/home/baliu/.venvs/qwen_ft/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


prediction with rationale
v3 means a version with rationale in Qwen

In [3]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --------------------------------------
# Config
# --------------------------------------

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale.csv")
# data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2

SEP = "=" * 80

MAX_PROMPT_CHARS = 30000
# MAX_PROMPT_CHARS = 30000  # Qwen 7B can handle up to 32k tokens, which is roughly 30k chars, avoid OOM

TIMEOUT_SEC = 600
COOLDOWN_EVERY = 20
COOLDOWN_SEC = 10

# --------------------------------------
# Timeout handling
# --------------------------------------

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# Load done users
# --------------------------------------
def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------
with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

# --------------------------------------
# Main loop
# --------------------------------------
try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        if len(prompt) > MAX_PROMPT_CHARS:
            print("⚠️ Prompt too long, skipping")
            continue

        try:
            signal.alarm(TIMEOUT_SEC)

            messages =[
                {"role": "system",
                 "content": (
                     "You are a mobility behavior and socioeconomic status inference analyst."
                     "return only valid JSON with keys: age_group, gender, household_size, household_income_level."
                     "Based only on the mobility behaviour and nearby spatial context described below,"
                     # "infer the user's most likely household income level based on swiss census data and situation\n\n"
                     "Please choose exactly one of the following categories (monthly household income in CHF),age_group, gender:\n"
                     "<4000, 4001-8000, 8001-12000, 12001-16000, 16001+. age: 0-17, 18-24, 25-34, 35-44, 45-54, 55-64, 65+. gender: male, female, other, perefer not to say\n\n"
                     " Guidelines:\n"
                     "- Use only the information explicitly provided in the mobility record.\n"
                     "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
                     "- Do not assume any unobserved personal attributes.\n"
                     "- keep a neutral and factual tone.\n\n"
                     "Output format:\n"
                     "1. Reasoning (no more than 150 words).\n" # need to adjust the parameters for rationale length
                     "2. One final line in strict JSON format, for example:\n"
                     " {\"household_income_level\": \"4001-8000\", \"age_group\": \"45-54\", \"gender\": \"other\", \"income_rationale\": [\"visited high-income neighborhoods\", \"used premium transport modes\", \"frequent visits to business districts\"], \"income_confidence\": \"high\"}\n\n"
                     "- income_rationale must be a list of 3-6 short bullet-like strings, each describing a concrete mobility/semantic evidence. \n "
                     "- Do not provide step by step reasoning, or hidden chain of thought"
                     " Just provide evidence-style justification.\n"
                     "- income_confidence must be one of :low, medium, high.\n"
                     "no extra keys, no markdown, no commentary outside josn"

                 )
                 },
            ]

            tokenizer = AutoTokenizer.from_pretrained(
                MODEL_NAME,
                trust_remote_code=True)

            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=512, # give space for rationale
                temperature=0.5,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.05 # to encourage more diverse outputs
            )

            signal.alarm(0)
            
            # decode only the generated part, avoid mixing prompt + answer
            gen_ids = outputs[0][inputs["input_ids"].shape[1]:]  

            decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            json_match = re.search(r"\{.*\}", decoded, re.DOTALL)
            if json_match:
                #json_str = json_match.group(0)
                #try:
                    #result = json.loads(json_str)


                # need to fix 
                result = json.loads(json_match.group(0))
            else:
                result = {"raw_output": decoded, "error": "json_parse_failed"}

            result["user_id"] = user_id

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done_users.add(user_id)
            print("✅ Done")

        except TimeoutError:
            print("⏰ Timeout")
            continue

        except Exception as e:
            print("❌ Error:", e)
            time.sleep(2)
            continue

        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted safely.")
    sys.exit(0)

# --------------------------------------
# Save csv snapshot
# --------------------------------------

if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved → {PRED_CSV}")
else:
    print("⚠️ No predictions generated")

📦 Loaded 2102 prompts
🔁 Found 1014 completed users
⏭️ Skipping AAGAF
⏭️ Skipping AAINS
⏭️ Skipping AAQME
⏭️ Skipping AARWP
⏭️ Skipping ABARC
⏭️ Skipping ABECR
⏭️ Skipping ABENL
⏭️ Skipping ABISR
⏭️ Skipping ABLPH
⏭️ Skipping ACSUF
⏭️ Skipping ACWVM
⏭️ Skipping ADCMH
⏭️ Skipping AEBWY
⏭️ Skipping AEGHJ
⏭️ Skipping AEWVV
⏭️ Skipping AFNCT
⏭️ Skipping AFUVG
⏭️ Skipping AGHUY
⏭️ Skipping AHTYK
⏭️ Skipping AJJJR
⏭️ Skipping AJJNX
⏭️ Skipping AJKBY
⏭️ Skipping AJQGV
⏭️ Skipping AKGPG
⏭️ Skipping AKMIL
⏭️ Skipping AKOZE
⏭️ Skipping AKQKS
⏭️ Skipping ALKPX
⏭️ Skipping ALSMT
⏭️ Skipping AMGUM
⏭️ Skipping AMGZL
⏭️ Skipping AMYMD
⏭️ Skipping ANEFF
⏭️ Skipping ANMKC
⏭️ Skipping ANQHI
⏭️ Skipping AOVGU
⏭️ Skipping AOVLF
⏭️ Skipping APKPN
⏭️ Skipping APWYR
⏭️ Skipping AQFEA
⏭️ Skipping AQQGE
⏭️ Skipping AQREK

🔮 [43/2102] Predicting ARFNG
✅ Done
⏭️ Skipping ARMZF
⏭️ Skipping ARZDI
⏭️ Skipping ASUKL
⏭️ Skipping ASXOD
⏭️ Skipping ATBSI
⏭️ Skipping ATCBP
⏭️ Skipping ATEAN
⏭️ Skipping ATKDY
⏭️ Skipping 

read prompts from text version

In [ ]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --------------------------------------
# Config
# --------------------------------------

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v2withrationale.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v2withrationale.csv")
# data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2

SEP = "=" * 80

MAX_PROMPT_CHARS = 25000
#MAX_PROMPT_CHARS = 30000 # Qwen 7B can handle up to 32k tokens, which is roughly 30k chars, but we set a safer limit to avoid OOM

TIMEOUT_SEC = 600
COOLDOWN_EVERY = 20
COOLDOWN_SEC = 10

# --------------------------------------
# GPU check
# --------------------------------------

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------
# Load Qwen 4bit
# --------------------------------------

print("🔄 Loading Qwen...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("✅ Qwen ready.")

# --------------------------------------
# Timeout handling
# --------------------------------------

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# Load done users
# --------------------------------------

def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------

with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

# --------------------------------------
# Main loop
# --------------------------------------

try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        if len(prompt) > MAX_PROMPT_CHARS:
            print("⚠️ Prompt too long, skipping")
            continue

        try:
            signal.alarm(TIMEOUT_SEC)

            messages = [
                {"role": "system",
                 "content": "Respond only in valid JSON with keys: age_group, gender, income_level."},
                {"role": "user", "content": prompt}
            ]

            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.3,
                do_sample=False
            )

            signal.alarm(0)

            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

            json_match = re.search(r"\{.*\}", decoded, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
            else:
                result = {"raw_output": decoded, "error": "json_parse_failed"}

            result["user_id"] = user_id

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done_users.add(user_id)
            print("✅ Done")

        except TimeoutError:
            print("⏰ Timeout")
            continue

        except Exception as e:
            print("❌ Error:", e)
            time.sleep(2)
            continue

        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted safely.")
    sys.exit(0)

# --------------------------------------
# Save csv snapshot
# --------------------------------------

if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved → {PRED_CSV}")
else:
    print("⚠️ No predictions generated")

CUDA: True
GPU: NVIDIA L4
🔄 Loading Qwen...


Loading weights: 100%|██████████| 339/339 [00:02<00:00, 138.25it/s, Materializing param=model.norm.weight]                              


✅ Qwen ready.
📦 Loaded 2102 prompts
🔁 Found 0 completed users

🔮 [1/2102] Predicting AAGAF
✅ Done

🔮 [2/2102] Predicting AAINS
✅ Done

🔮 [3/2102] Predicting AAQME
✅ Done

🔮 [4/2102] Predicting AARWP
✅ Done

🔮 [5/2102] Predicting ABARC
✅ Done

🔮 [6/2102] Predicting ABECR
✅ Done

🔮 [7/2102] Predicting ABENL
✅ Done

🔮 [8/2102] Predicting ABISR
✅ Done

🔮 [9/2102] Predicting ABLPH
✅ Done

🔮 [10/2102] Predicting ACSUF
✅ Done

🔮 [11/2102] Predicting ACWVM
✅ Done

🔮 [12/2102] Predicting ADCMH
✅ Done

🔮 [13/2102] Predicting AEBWY
✅ Done

🔮 [14/2102] Predicting AEGHJ
✅ Done

🔮 [15/2102] Predicting AEWVV
✅ Done

🔮 [16/2102] Predicting AFNCT
✅ Done

🔮 [17/2102] Predicting AFUVG
✅ Done

🔮 [18/2102] Predicting AGHUY
✅ Done

🔮 [19/2102] Predicting AHTYK
✅ Done

🔮 [20/2102] Predicting AJJJR
✅ Done
🧹 Cooling down GPU...

🔮 [21/2102] Predicting AJJNX
✅ Done

🔮 [22/2102] Predicting AJKBY
✅ Done

🔮 [23/2102] Predicting AJQGV
✅ Done

🔮 [24/2102] Predicting AKGPG
✅ Done

🔮 [25/2102] Predicting AKMIL
✅ Done


## Evaluation

In [4]:
import pandas as pd
import json
import re
from pathlib import Path

# --------------------------------------
# Load raw predictions
# --------------------------------------
raw_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale.csv")
# preds_qwen_income_22Feb2026_v4withrationale
#preds_qwen_income_22Feb2026_v2withrationale.csv
# # preds_qwen_income_22Feb2026_v5withrationale

df = pd.read_csv(raw_path)
print(f"Loaded raw predictions: {len(df)} rows")

# --------------------------------------
# Income extraction functions
# --------------------------------------
def extract_from_text(text):
    if pd.isna(text):
        return None

    try:
        data = json.loads(text)
        income_level = data.get("income_level", "")
        if income_level:
            if income_level in [">=16000", ">=16001", "16001+", "16000+", "more than 16000", "above 16000", "at least 16000", "16000+"]:
                return ">16000"
            return income_level
    except Exception:
        pass

    text = str(text).lower()

    norm = (
        text.lower()
            .replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    if re.search(
        r'(>=\s*16000|>=\s*16001|16001\+|16000\+|'
        r'more\s+than\s+16000|above\s+16000|at\s+least\s+16000)',
        norm
    ):
        return ">16000"

    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}[,.]?\d{3})\s*[-–]\s*(\d{1,2}[,.]?\d{3})', norm)

    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    else:
        return None

# --------------------------------------
# Apply extraction
# --------------------------------------
TEXT_COL = "household_income_level"
df["household_income_level"] = df[TEXT_COL].apply(extract_from_text)

print(df.columns.tolist())
print(df.head(3))
print(df["household_income_level"].value_counts(dropna=False))

# --------------------------------------
# Save cleaned CSV
# -----------------------------------------
df_clean = (
    df[["user_id", "household_income_level"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.jsonl")
df_clean.to_json(out_path, orient="records", lines=True)
out_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv")
# preds_qwen_income_22Feb2026_v2withrationale.csv
df_clean.to_csv(out_path, index=False)

print(f"✅ Clean predictions saved to: {out_path}")
print(f"📊 Rows: {len(df_clean)}")
print(df_clean.head(10))

Loaded raw predictions: 2062 rows
['household_income_level', 'age_group', 'gender', 'income_rationale', 'income_confidence', 'user_id']
  household_income_level age_group  gender  \
0            12001-16000     35-44  female   
1            12001-16000     35-44  female   
2             8001-12000     25-34  female   

                                    income_rationale income_confidence user_id  
0  ['frequent visits to high-cost business areas'...              high   AAGAF  
1  ['visited multiple high-cost leisure and enter...              high   AAINS  
2  ['frequent visits to middle to high-income nei...            medium   AAQME  
household_income_level
8001-12000     1206
12001-16000     856
Name: count, dtype: int64
✅ Clean predictions saved to: /data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv
📊 Rows: 2062
  user_id household_income_level
0   AAGAF            12001-16000
1   AAINS            12001-16000
2   AAQME             8001-12000
3   AARW

In [14]:
import pandas as pd
import json
import re
from pathlib import Path

# --------------------------------------
# Load raw predictions
# --------------------------------------
raw_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale.csv")
df = pd.read_csv(raw_path)
print(f"Loaded raw predictions: {len(df)} rows")

# --------------------------------------
# Income bin mapping
# --------------------------------------
def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    return None

# --------------------------------------
# Generic JSON safe loader
# --------------------------------------
def safe_json_load(text):
    if pd.isna(text):
        return {}
    try:
        return json.loads(text)
    except Exception:
        return {}

# --------------------------------------
# Gender cleaning
# --------------------------------------
def clean_gender(text):
    text = str(text).lower()
    if "female" in text or text == "f":
        return "female"
    if "male" in text or text == "m":
        return "male"
    return None

# --------------------------------------
# Age group cleaning
# --------------------------------------
VALID_AGE_GROUPS = [
    "18-25", "26-35", "36-45", "46-55", "56-65", "66+"
]

def clean_age(text):
    if pd.isna(text):
        return None

    text = str(text).lower()

    for group in VALID_AGE_GROUPS:
        if group in text:
            return group

    # catch patterns like "between 26 and 35"
    m = re.search(r'(\d{2})\s*[-–]\s*(\d{2})', text)
    if m:
        return f"{m.group(1)}-{m.group(2)}"

    return None

# --------------------------------------
# Income cleaning
# --------------------------------------
def clean_income(text):
    if pd.isna(text):
        return None

    # Try JSON first
    try:
        data = json.loads(text)
        income = data.get("income_level", "")
        if income:
            income = str(income)
            if "16000" in income or "16001" in income:
                return ">16000"
            return income
    except Exception:
        pass

    text = str(text).lower()
    norm = (
        text.replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    # high income
    if re.search(r'(>=\s*16000|16000\+|more\s+than\s+16000|above\s+16000)', norm):
        return ">16000"

    # between pattern
    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}\d{3})\s*[-–]\s*(\d{1,2}\d{3})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    # direct bin mention
    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

# --------------------------------------
# Apply extraction
# --------------------------------------

def extract_all(text):
    data = safe_json_load(text)

    # Try JSON fields first
    age = data.get("age_group", None)
    gender = data.get("gender", None)
    income = data.get("income_level", None)

    # Fallback to raw text parsing
    if age is None:
        age = clean_age(text)
    else:
        age = clean_age(age)

    if gender is None:
        gender = clean_gender(text)
    else:
        gender = clean_gender(gender)

    if income is None:
        income = clean_income(text)
    else:
        income = clean_income(income)

    return pd.Series([age, gender, income])

TEXT_COL = "household_income_level"

df[["age_group", "gender", "household_income_level"]] = \
    df[TEXT_COL].apply(extract_all)

print("\nValue counts:")
print("Age:")
print(df["age_group"].value_counts(dropna=False))
print("\nGender:")
print(df["gender"].value_counts(dropna=False))
print("\nIncome:")
print(df["household_income_level"].value_counts(dropna=False))

# --------------------------------------
# Save cleaned data
# --------------------------------------
df_clean = (
    df[["user_id", "age_group", "gender", "household_income_level"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_csv = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5_clean_full.csv")
df_clean.to_csv(out_csv, index=False)

out_jsonl = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5_clean_full.jsonl")
df_clean.to_json(out_jsonl, orient="records", lines=True)

print(f"\n✅ Clean predictions saved to: {out_csv}")
print(f"📊 Rows: {len(df_clean)}")
print(df_clean.head(10))

Loaded raw predictions: 2062 rows

Value counts:
Age:
age_group
01-12    1206
01-16     856
Name: count, dtype: int64

Gender:
gender
NaN    2062
Name: count, dtype: int64

Income:
household_income_level
8001-12000     1206
12001-16000     856
Name: count, dtype: int64

✅ Clean predictions saved to: /data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5_clean_full.csv
📊 Rows: 2062
  user_id age_group  gender household_income_level
0   AAGAF     01-16     NaN            12001-16000
1   AAINS     01-16     NaN            12001-16000
2   AAQME     01-12     NaN             8001-12000
3   AARWP     01-16     NaN            12001-16000
4   ABARC     01-16     NaN            12001-16000
5   ABECR     01-16     NaN            12001-16000
6   ABENL     01-16     NaN            12001-16000
7   ABISR     01-16     NaN            12001-16000
8   ABLPH     01-12     NaN             8001-12000
9   ACSUF     01-12     NaN             8001-12000


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------
# File paths
# --------------------------------------
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv")
# preds_qwen_income_22Feb2026_v2withrationale_clean

# --------------------------------------
# Load data
# --------------------------------------
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

# --------------------------------------
# Select & align columns
# --------------------------------------
gt = gt[["user_id", "age_group", "household_size_group", "income_level"]]
# pred = pred[["user_id", "age_group", "household_sizes", "income_level"]]
pred = pred[["user_id","age_group", "gender", "household_income_level"]]

# --------------------------------------
# Normalize household size
# --------------------------------------
# def normalize_household_size(x):
#     if pd.isna(x):
#         return np.nan
#     try:
#         x = float(x)
#     except Exception:
#         return np.nan

#     if x >= 5:
#         return "5+"
#     else:
#         return str(int(x))

# gt["hh_norm"] = gt["household_size_group"].apply(normalize_household_size)
# pred["hh_norm"] = pred["household_sizes"].apply(normalize_household_size)

# --------------------------------------
# Merge GT and predictions
# --------------------------------------
merged = pd.merge(
    gt,
    pred,
    on="user_id",
    suffixes=("_true", "_pred")
)

print(f"Merged {len(merged)} users")
print(merged.head(5))

# --------------------------------------
# Masks for valid evaluation samples
# --------------------------------------
age_mask = (
     merged["age_group_true"].notna() &
     merged["age_group_pred"].notna()
)

hh_mask = (
    merged["hh_norm_true"].notna() &
    merged["hh_norm_pred"].notna()
)

income_mask = (
   merged["income_level_true"].notna() &
   merged["income_level_pred"].notna() &
  (merged["income_level_true"].str.lower() != "prefer not to say")
)

income_mask = (
    merged["household_income_level"].notna() &
    merged["income_level"].notna() &
    (merged["household_income_level"].str.lower() != "prefer not to say")
)
print("\nEvaluation sample sizes:")
print(f"Age:        {age_mask.sum()}")
print(f"Household:  {hh_mask.sum()}")
print(f"Income:     {income_mask.sum()}")

# --------------------------------------
# Accuracy computation
# # --------------------------------------
age_acc = (
    merged.loc[age_mask, "age_group_true"]
    == merged.loc[age_mask, "age_group_pred"]
).mean()

household_acc = (
    merged.loc[hh_mask, "hh_norm_true"]
    == merged.loc[hh_mask, "hh_norm_pred"] 
    ).mean()

income_acc = (
    merged.loc[income_mask, "household_income_level"]
    == merged.loc[income_mask, "income_level"]
).mean()

# --------------------------------------
# Majority-class baselines (for context)
# --------------------------------------
def majority_baseline(y_true):
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

age_baseline = majority_baseline(
    merged.loc[age_mask, "age_group_true"]
)

hh_baseline = majority_baseline(
    merged.loc[hh_mask, "hh_norm_true"]
)

income_baseline = majority_baseline(

    merged.loc[income_mask, "household_income_level"]
)

# --------------------------------------
# Output results
# --------------------------------------
print("\n📊 Accuracy Results:")
# print(f"Age group accuracy:        {age_acc:.3f} (baseline {age_baseline:.3f})")
# print(f"Household size accuracy:  {household_acc:.3f} (baseline {hh_baseline:.3f})")
#print(f"Income level accuracy:    {income_acc:.3f} (baseline {income_baseline:.3f})")
print(f"Income level accuracy:    {income_acc:.3f} ")
# --------------------------------------
# Save merged comparison tables
# --------------------------------------
output_path = Path("/data/baliu/python_code/data/merged_comparison_v6_final_qwen_v5.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path(
    "/data/baliu/python_code/data/comparison_income_eval_v6_final_qwen_v5.csv"
)
merged.loc[income_mask].to_csv(
    filtered_income_path, index=False, encoding="utf-8"
)

print(f"\nDetailed comparison saved to: {output_path}")
print(f"Income-eval subset saved to: {filtered_income_path}")

# --------------------------------------
# Confusion matrices
# --------------------------------------
age_cm = pd.crosstab(
    merged.loc[age_mask, "age_group_true"],
    merged.loc[age_mask, "age_group_pred"],
    normalize="index"
)

age_cm_path = Path(
   "/data/baliu/python_code/data/age_confusion_matrix_v4_final.csv"
)
age_cm.to_csv(age_cm_path)

print(f"Age confusion matrix saved to: {age_cm_path}")

# ---- Confusion matrix (income) ----
income_cm = pd.crosstab(
    merged.loc[income_mask, "income_level"],
    merged.loc[income_mask, "household_income_level"]
    #normalize="index"
)

print("\nIncome confusion matrix (counts):")
print(income_cm)

income_cm = pd.crosstab(
    merged.loc[income_mask, "income_level"],
    merged.loc[income_mask, "household_income_level"],
    normalize="index"
)

print("\nIncome confusion matrix (row-normalized):")
print(income_cm)

/tmp/ipykernel_278062/254681516.py:15: DtypeWarning: Columns (0: postcode_home, 1: education, 2: main_employment, 3: income, 4: household_size, 5: citizen_1, 6: citizen_2, 7: intro_survey_started_at, 8: intro_survey_finished_at, 9: intro_survey_finished, 10: work_status_employed, 11: work_status_self_employed, 12: work_status_unemployed, 13: work_status_apprentice, 14: work_status_student, 15: work_status_retired, 16: work_status_other, 17: own_vehicles_car, 18: own_vehicles_motorbike, 19: own_vehicles_bicycle, 20: car_fuel, 21: car_year, 22: car_size, 23: bike_type_regular, 24: bike_type_ebike_25, 25: bike_type_ebike_45, 26: pt_pass_ga, 27: pt_pass_half_fare, 28: pt_pass_regional_pass, 29: pt_pass_track_7, 30: pt_pass_other, 31: pt_pass_no_pass, 32: freq_cardriver_own_car, 33: freq_cardriver_shared_car, 34: freq_carpass_car_in_hh, 35: freq_carpass_car_pooling, 36: freq_carpass_taxi, 37: freq_carpass_app_based, 38: freq_pt_train, 39: freq_pt_local_pt, 40: freq_bike_own_bike, 41: freq_b

Loaded ground truth (90909 rows)
Loaded predictions  (2062 rows)


KeyError: "['age_group', 'gender'] not in index"

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------
# File paths
# --------------------------------------
gt_path   = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv")

# --------------------------------------
# Load data
# --------------------------------------
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

# --------------------------------------
# Select & align columns
# --------------------------------------
gt = gt[["user_id", "age_group", "household_size_group", "income_level"]].copy()
pred = pred[["user_id", "age_group", "gender", "household_income_level"]].copy()

# --------------------------------------
# Merge GT and predictions
# --------------------------------------
merged = pd.merge(gt, pred, on="user_id", how="inner", suffixes=("_true", "_pred"))
print(f"Merged {len(merged)} users")
print(merged.head(5))

# --------------------------------------
# Masks for valid evaluation samples
# --------------------------------------
age_mask = merged["age_group_true"].notna() & merged["age_group_pred"].notna()

# If you don't evaluate household size right now, set hh_mask to all-False safely:
hh_mask = pd.Series(False, index=merged.index)

# Income: evaluate where both exist, and GT is not "prefer not to say"
income_mask = (
    merged["income_level_true"].notna()
    & merged["household_income_level"].notna()
    & (merged["income_level_true"].astype(str).str.lower() != "prefer not to say")
)

print("\nEvaluation sample sizes:")
print(f"Age:        {age_mask.sum()}")
print(f"Household:  {hh_mask.sum()}")
print(f"Income:     {income_mask.sum()}")

# --------------------------------------
# Accuracy computation
# --------------------------------------
age_acc = (
    merged.loc[age_mask, "age_group_true"]
    == merged.loc[age_mask, "age_group_pred"]
).mean() if age_mask.any() else np.nan

income_acc = (
    merged.loc[income_mask, "income_level_true"]
    == merged.loc[income_mask, "household_income_level"]
).mean() if income_mask.any() else np.nan

# --------------------------------------
# Majority-class baselines (TRUE labels only)
# --------------------------------------
def majority_baseline(y_true: pd.Series) -> float:
    y_true = y_true.dropna()
    if len(y_true) == 0:
        return np.nan
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

age_baseline = majority_baseline(merged.loc[age_mask, "age_group_true"])
income_baseline = majority_baseline(merged.loc[income_mask, "income_level_true"])

# --------------------------------------
# Output results
# --------------------------------------
print("\n📊 Accuracy Results:")
print(f"Age group accuracy:     {age_acc:.3f} (baseline {age_baseline:.3f})")
print(f"Income level accuracy:  {income_acc:.3f} (baseline {income_baseline:.3f})")

# --------------------------------------
# Save merged comparison tables
# --------------------------------------
output_path = Path("/data/baliu/python_code/data/merged_comparison_v6_final_qwen_v5.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path("/data/baliu/python_code/data/comparison_income_eval_v6_final_qwen_v5.csv")
merged.loc[income_mask].to_csv(filtered_income_path, index=False, encoding="utf-8")

print(f"\nDetailed comparison saved to: {output_path}")
print(f"Income-eval subset saved to: {filtered_income_path}")

# --------------------------------------
# Confusion matrices
# --------------------------------------
# Age confusion matrix (row-normalized)
if age_mask.any():
    age_cm = pd.crosstab(
        merged.loc[age_mask, "age_group_true"],
        merged.loc[age_mask, "age_group_pred"],
        normalize="index"
    )
    age_cm_path = Path("/data/baliu/python_code/data/age_confusion_matrix_v5_final.csv")
    age_cm.to_csv(age_cm_path)
    print(f"Age confusion matrix saved to: {age_cm_path}")

# Income confusion matrix (counts + row-normalized)
if income_mask.any():
    income_cm_counts = pd.crosstab(
        merged.loc[income_mask, "income_level_true"],
        merged.loc[income_mask, "household_income_level"]
    )
    print("\nIncome confusion matrix (counts):")
    print(income_cm_counts)

    income_cm_norm = pd.crosstab(
        merged.loc[income_mask, "income_level_true"],
        merged.loc[income_mask, "household_income_level"],
        normalize="index"
    )
    print("\nIncome confusion matrix (row-normalized):")
    print(income_cm_norm)

/tmp/ipykernel_278062/1828059493.py:14: DtypeWarning: Columns (0: postcode_home, 1: education, 2: main_employment, 3: income, 4: household_size, 5: citizen_1, 6: citizen_2, 7: intro_survey_started_at, 8: intro_survey_finished_at, 9: intro_survey_finished, 10: work_status_employed, 11: work_status_self_employed, 12: work_status_unemployed, 13: work_status_apprentice, 14: work_status_student, 15: work_status_retired, 16: work_status_other, 17: own_vehicles_car, 18: own_vehicles_motorbike, 19: own_vehicles_bicycle, 20: car_fuel, 21: car_year, 22: car_size, 23: bike_type_regular, 24: bike_type_ebike_25, 25: bike_type_ebike_45, 26: pt_pass_ga, 27: pt_pass_half_fare, 28: pt_pass_regional_pass, 29: pt_pass_track_7, 30: pt_pass_other, 31: pt_pass_no_pass, 32: freq_cardriver_own_car, 33: freq_cardriver_shared_car, 34: freq_carpass_car_in_hh, 35: freq_carpass_car_pooling, 36: freq_carpass_taxi, 37: freq_carpass_app_based, 38: freq_pt_train, 39: freq_pt_local_pt, 40: freq_bike_own_bike, 41: freq_

Loaded ground truth (90909 rows)
Loaded predictions  (2062 rows)


KeyError: "['age_group', 'gender'] not in index"